# Strategy Document

The second price pattern I noticed is for strong fundamental stocks. Here is the setup

* Both revenue and EPS has double growth over at least 2 digits.
* Price trend relatively strong.
* The business either has a Economic Moat, or link to a strong narrative of the whole market. like NVDA with economic moat, ANET link to the AI infra building.
* It is not the end of the cycle. Typically a new cycle growth will take at least two year.

For such stock, can just buy and hold until the cycle near peak which need keep track the expectation change from earning report history. The key part is to follow the industry and expectation release during the conference call.

In [1]:
import datetime as dt
import pandas as pd
import numpy as np
from sts.dio.equity import Ticker


class GrowthHalo:
    def __init__(
        self,
        symbol: str,
        high_growth_threshold: float = 0.2,
        medium_growth_ratio: float = 0.1,
        yearly_growth_sharpe: float = 2.0,
        quarterly_growth_sharpe: float = 1.0,
    ):
        self.symbol = symbol
        self.ticker = Ticker(symbol)
        self.high_growth_threshold = high_growth_threshold
        self.medium_growth_ratio = medium_growth_ratio
        self.yearly_growth_sharpe = yearly_growth_sharpe
        self.quarterly_growth_sharpe = quarterly_growth_sharpe

        self.init()

    def init(self):
        self.statement_quarterly = self.ticker.get_income_stmt(freq="quarterly").T.sort_index()
        self.statement_yearly = self.ticker.get_income_stmt(freq="yearly").T.sort_index()

        start = self.statement_yearly.index[0].strftime("%Y-%m-%d")
        end = dt.datetime.today().strftime("%Y-%m-%d")
        self.price_hist = self.ticker.history(start=start, end=end, interval="1d")

        self.statement_quarterly = pd.merge_asof(
            self.statement_quarterly,
            self.price_hist[["Close"]],
            left_index=True,
            right_index=True,
        )
        self.statement_yearly = pd.merge_asof(
            self.statement_yearly,
            self.price_hist[["Close"]],
            left_index=True,
            right_index=True,
        )

        self.statement_quarterly["PE"] = (
            self.statement_quarterly["Close"] / self.statement_quarterly["DilutedEPS"] / 4.0
        )
        self.statement_yearly["PE"] = self.statement_yearly["Close"] / self.statement_yearly["DilutedEPS"]

    def __repr__(self):
        return f"GrowthHalo for ticker: %s" % (self.symbol)

    @property
    def EPS_quarterly(self):
        return self.statement_quarterly["DilutedEPS"]

    @property
    def EPS_yearly(self):
        return self.statement_yearly["DilutedEPS"]

    @property
    def OperatingRevenue_quarterly(self):
        return self.statement_quarterly["OperatingRevenue"]

    @property
    def OperatingRevenue_yearly(self):
        return self.statement_yearly["OperatingRevenue"]

    @property
    def eps_yearly_growth(self):
        return self.EPS_yearly / self.EPS_yearly.shift() - 1.0

    @property
    def eps_quarterly_growth(self):
        return self.EPS_quarterly / self.EPS_quarterly.shift() - 1.0

    @property
    def OperatingRevenue_yearly_growth(self):
        return self.OperatingRevenue_yearly / self.OperatingRevenue_yearly.shift() - 1.0

    @property
    def OperatingRevenue_quarterly_growth(self):
        return self.OperatingRevenue_quarterly / self.OperatingRevenue_quarterly.shift() - 1.0

    @property
    def pe_yearly(self):
        return self.statement_yearly["PE"]

    @property
    def pe_quarterly(self):
        return self.statement_quarterly["PE"]

    def sharpe_mod_ratio(self, growth_history: pd.Series) -> float:
        mean = growth_history.mean()
        diff = np.minimum(0, growth_history.diff().dropna())  # only care about downside move
        std = np.sqrt(np.mean(diff * diff))
        if std < 1e-8:
            return 100
        else:
            return mean / std

    def get_growth_aspect(self):
        """
        the bigger the number, the better
        """
        eps_growth_yearly = self.eps_yearly_growth.dropna()
        eps_growth_quarterly = self.eps_quarterly_growth.dropna() * 4.0  # scale to annual number

        if (
            (eps_growth_yearly.mean() >= self.high_growth_threshold)
            and (self.sharpe_mod_ratio(eps_growth_quarterly) >= self.quarterly_growth_sharpe)
            and np.all(eps_growth_quarterly > 0)
        ):
            if eps_growth_quarterly.mean() >= self.high_growth_threshold:  # still on expension
                return 2.0
            elif eps_growth_quarterly.mean() >= self.medium_growth_ratio:  # around the peak of the cycle
                return 1.0
            else:
                return 0.0  # typically means the end of the cycle
        else:
            return 0.0

In [2]:
from sts.dio.symbols import sp500_symbol_table
from sts.utils.logging import get_logger, logging

logger = get_logger("GrowthHalo", logging.INFO)

In [3]:
growth_scan_df = None
for sym in sp500_symbol_table.to_symbol():
    try:
        logger.info(sym)
        sym_growth_halo = GrowthHalo(sym.YahooSym, yearly_growth_sharpe=1.5, quarterly_growth_sharpe=0.8)
        if sym_growth_halo.get_growth_aspect() > 0:
            logger.info("adding %s" % (sym.YahooSym))
            df2 = sym_growth_halo.eps_yearly_growth.dropna()
            df3 = sym_growth_halo.pe_yearly.dropna()
            df = pd.DataFrame(
                {
                    "sym": sym.YahooSym,
                    "EPS": df2,
                    "PE": df3,
                    "GrowthHalo": sym_growth_halo,
                }
            )
            growth_scan_df = pd.concat([growth_scan_df, df])
    except Exception as e:
        logger.error(str(e))

2024-11-20 21:27:21,615 - GrowthHalo - INFO - MMM
2024-11-20 21:27:22,440 - GrowthHalo - INFO - AOS
2024-11-20 21:27:23,076 - GrowthHalo - INFO - ABT
2024-11-20 21:27:23,739 - GrowthHalo - INFO - ABBV
2024-11-20 21:27:24,429 - GrowthHalo - INFO - ACN
2024-11-20 21:27:24,998 - GrowthHalo - INFO - ATVI
2024-11-20 21:27:25,236 - GrowthHalo - ERROR - index 0 is out of bounds for axis 0 with size 0
2024-11-20 21:27:25,238 - GrowthHalo - INFO - ADM
2024-11-20 21:27:25,953 - GrowthHalo - INFO - ADBE
2024-11-20 21:27:26,550 - GrowthHalo - INFO - ADP
2024-11-20 21:27:27,250 - GrowthHalo - INFO - AAP
2024-11-20 21:27:27,846 - GrowthHalo - INFO - AES
2024-11-20 21:27:28,454 - GrowthHalo - INFO - AFL
2024-11-20 21:27:29,068 - GrowthHalo - INFO - A
2024-11-20 21:27:29,693 - GrowthHalo - INFO - APD
2024-11-20 21:27:30,286 - GrowthHalo - INFO - AKAM
2024-11-20 21:27:30,942 - GrowthHalo - INFO - ALK
2024-11-20 21:27:31,620 - GrowthHalo - INFO - ALB
2024-11-20 21:27:32,280 - GrowthHalo - INFO - ARE
202

In [5]:
min_growth_sym = growth_scan_df.groupby("sym")["EPS"].min()
min_growth_sym[min_growth_sym > 0]

sym
ANET    0.316646
IT      0.081433
SBAC    0.092417
URI     0.189882
Name: EPS, dtype: object

In [6]:
growth_scan_df

,sym,EPS,PE,GrowthHalo
2020-12-31,ALLE,NaN,32.651013,GrowthHalo for ticker: ALLE
2021-12-31,ALLE,0.575221,23.848694,GrowthHalo for ticker: ALLE
2022-12-31,ALLE,-0.02809,19.738465,GrowthHalo for ticker: ALLE
2023-12-31,ALLE,0.179191,20.474453,GrowthHalo for ticker: ALLE
2020-12-31,ANET,NaN,36.366709,GrowthHalo for ticker: ANET
2021-12-31,ANET,0.316646,54.657795,GrowthHalo for ticker: ANET
2022-12-31,ANET,0.623574,28.419203,GrowthHalo for ticker: ANET
2023-12-31,ANET,0.540984,35.791792,GrowthHalo for ticker: ANET
2020-12-31,IT,NaN,54.118244,GrowthHalo for ticker: IT
2021-12-31,IT,2.111486,36.299675,GrowthHalo for ticker: IT


In [43]:
growth_scan_df = growth_scan_df[growth_scan_df["sym"].isin(min_growth_sym[min_growth_sym > 0].index)]

In [44]:
growth_scan_df

,sym,EPS,PE,GrowthHalo
2020-12-31,ANET,NaN,36.366709,GrowthHalo for ticker: ANET
2021-12-31,ANET,0.316646,54.657795,GrowthHalo for ticker: ANET
2022-12-31,ANET,0.623574,28.419203,GrowthHalo for ticker: ANET
2023-12-31,ANET,0.540984,35.791792,GrowthHalo for ticker: ANET
2020-12-31,FTNT,NaN,51.21724,GrowthHalo for ticker: FTNT
2021-12-31,FTNT,0.258621,98.46575,GrowthHalo for ticker: FTNT
2022-12-31,FTNT,0.452055,46.122641,GrowthHalo for ticker: FTNT
2023-12-31,FTNT,0.377358,40.08904,GrowthHalo for ticker: FTNT
2020-12-31,RSG,NaN,30.273367,GrowthHalo for ticker: RSG
2021-12-31,RSG,0.337748,33.27242,GrowthHalo for ticker: RSG


In [23]:
self = growth_scan_df.iloc[0]["GrowthHalo"]

In [45]:
self = GrowthHalo("NVDA", yearly_growth_sharpe=1.5, quarterly_growth_sharpe=0.8)
eps_growth_yearly = self.eps_yearly_growth.dropna()
eps_growth_quarterly = self.eps_quarterly_growth.dropna() * 4.0  # scale to annual number

In [48]:
eps_growth_yearly

2022-01-31    1.231884
2023-01-31   -0.548052
2024-01-31    5.856322
Name: DilutedEPS, dtype: object

In [49]:
eps_growth_quarterly

2023-07-31    8.195122
2023-10-31       1.936
2024-07-31    0.481605
Name: DilutedEPS, dtype: object

In [46]:
self.sharpe_mod_ratio(eps_growth_quarterly)

0.7785535066322627

In [47]:
self.sharpe_mod_ratio(eps_growth_yearly)

1.7321174063534965